<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/day4/notebooks/4_llm_annotation_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 4: Annotating Text with an LLM · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a
short comment. LLM output depends on the model version and the hardware, so read the comments as
"what to look for", not as exact results.

In this notebook an LLM codes US bill titles by policy area: the same task and the same human
labels as on Day 3, so you can compare with the TF-IDF baseline and fine-tuned BERT. We ask the
same small open model, **Qwen2.5 1.5B Instruct**, in five different ways and check each one
against the human coders:

| | Strategy | What changes |
|---|---|---|
| 1 | **Zero-shot** | only the task and the list of categories |
| 2 | **Codebook** | a definition for each category, plus a rule for edge cases |
| 3 | **Few-shot** | the codebook plus a handful of coded examples |
| 4 | **Chain-of-thought** | the model explains its reasoning before it answers |
| 5 | **Label scoring** | we read the model's probability for each category instead of letting it write |

Then we check whether the model agrees with itself, and, if there's time, run the same codebook
through **Ollama** and a much bigger model on **OpenRouter**.

### How this notebook is organized

| Section | | Rough time |
|---|---|---|
| 0–2. Setup, data, and the model | core | 10 min |
| 3–7. Five ways to ask | core | 30 min |
| 8. Comparing the strategies | core | 10 min |
| 9. Does the model agree with itself? | core | 5 min |
| 10. Other routes: Ollama and OpenRouter | optional | 15 min |

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

## 0. Setup

In [ ]:
!pip install -q transformers accelerate openai ollama

In [ ]:
import os
import time
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
import transformers
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from sklearn.metrics import accuracy_score, f1_score, classification_report, cohen_kappa_score, ConfusionMatrixDisplay

SEED = 42
set_seed(SEED)
pd.set_option("display.max_colwidth", 120)

print("transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → T4 GPU")

---
## 1. The data: bill titles coded by humans

The same data as on Day 3: titles of US Congressional bills, with the policy area that the
Congressional Research Service assigned. We use the same four areas.

In [ ]:
parts = []
for chunk in [0, 5]:          # two of the ten Parquet files, as on Day 3
    path = hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                           revision="refs/convert/parquet", filename=f"default/train/{chunk:04d}.parquet")
    parts.append(pd.read_parquet(path, columns=["title", "policy_area"]))
raw = pd.concat(parts, ignore_index=True)

LABELS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]

bills = (raw.assign(title=raw["title"].fillna("").str.strip())
            .query("title != ''")
            .drop_duplicates(subset="title"))
bills = bills[bills["policy_area"].isin(LABELS)].sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"{len(bills):,} titles in the four areas")

### Two samples, for two different jobs

An LLM needs no training data, but you still need **human-coded texts to check it on**. We set
aside two small samples:

- a **development set** (10 titles per area) to try out prompts and read the mistakes, and
- a **test set** (25 titles per area) that we use only to compare the final strategies.

Why two? Every prompt you try and keep *because it scored well* is a choice fitted to those
texts. If you tried ten prompts on the test set and reported the best, the score would be too
optimistic. The rest of the titles are a **pool** we can draw coded examples from.

In [ ]:
dev = bills.groupby("policy_area").head(10)
rest = bills.drop(dev.index)
test = rest.groupby("policy_area").head(25)
pool = rest.drop(test.index)
dev, test, pool = (d.reset_index(drop=True) for d in (dev, test, pool))

print(f"development: {len(dev)}   test: {len(test)}   pool of examples: {len(pool):,}")
test.sample(6, random_state=1)

---
## 2. The annotator

The same model as in the slides, loaded with Hugging Face `transformers`. `chat()` sends a list of
messages and returns the answer. `to_label()` turns whatever the model wrote into one of our four
categories, or `INVALID` if the answer doesn't name exactly one.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="auto")


def chat(messages, max_new_tokens=15, temperature=0.0):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    settings = {"do_sample": True, "temperature": temperature} if temperature > 0 else {"do_sample": False}
    with torch.no_grad():
        output = model.generate(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **settings)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def to_label(answer):
    text = answer.split("Label:")[-1].lower()          # for reasoning answers: only what follows 'Label:'
    found = [lab for lab in LABELS if lab.lower() in text]
    if not found and ("armed forces" in text or "national security" in text):
        found = ["Armed Forces and National Security"]
    return found[0] if len(found) == 1 else "INVALID"


print(to_label("Taxation"), "|", to_label("The answer is health."), "|", to_label("Health or Education?"))

One helper runs a strategy over a set of titles, times it, and scores it against the human
labels. Every result on the **test** set is stored in `results` for Section 8.

In [ ]:
results = []


def score(name, df, labels, seconds):
    return {
        "strategy": name,
        "accuracy": accuracy_score(df["policy_area"], labels),
        "macro-F1": f1_score(df["policy_area"], labels, labels=LABELS, average="macro", zero_division=0),
        "invalid answers": (labels == "INVALID").mean(),
        "seconds per title": seconds / len(df),
    }


def annotate(df, make_messages, name, max_new_tokens=15, temperature=0.0, ask=None, keep=True):
    ask = ask or (lambda m: chat(m, max_new_tokens, temperature))
    start = time.time()
    answers = pd.Series([ask(make_messages(t)) for t in df["title"]], index=df.index)
    labels = answers.map(to_label)
    row = score(name, df, labels, time.time() - start)
    if keep:
        results.append(row)
    print(f"{name}: accuracy {row['accuracy']:.2f} · macro-F1 {row['macro-F1']:.2f} · "
          f"invalid {row['invalid answers']:.0%} · {row['seconds per title']:.2f} s per title")
    return answers, labels

---
## 3. Zero-shot: just ask

The task and the four category names, nothing more. The instructions go in the **system**
message, the title in the **user** message.

In [ ]:
def zero_shot(title):
    system = ("You code US bill titles by policy area. Choose exactly one of: "
              + ", ".join(LABELS) + ". Reply with the category name only.")
    return [{"role": "system", "content": system},
            {"role": "user", "content": f"Bill title: {title}"}]


example = test["title"][0]
print(example, "→", chat(zero_shot(example)))

In [ ]:
zs_answers, zs_labels = annotate(test, zero_shot, "1 Zero-shot")

What did the model actually write? The raw answers next to the human labels:

In [ ]:
pd.DataFrame({"title": test["title"], "human": test["policy_area"],
              "model wrote": zs_answers, "label": zs_labels}).head(10)

> **✏️ Exercise 1**
>
> Run zero-shot on the **development** set (`annotate(dev, zero_shot, "...", keep=False)`), and
> look at the titles it got wrong. Which areas does it mix up? Do you see a pattern that a rule in
> a codebook could fix?

In [ ]:
# ✅ Solution
dev_answers, dev_labels = annotate(dev, zero_shot, "Zero-shot (development set)", keep=False)

wrong = dev.assign(model=dev_labels)[dev_labels != dev["policy_area"]]
print(pd.crosstab(wrong["policy_area"], wrong["model"]), "\n")
wrong[["title", "policy_area", "model"]]

# What to look for: errors usually cluster. Titles about veterans are a classic: the model often
# says Health (for veterans' health care) or Education (for veterans' education benefits), while
# the Congressional Research Service files veterans' issues under Armed Forces and National
# Security. Tax breaks for something else (college tuition, medical expenses) are another: the
# human coders tend to file anything amending the tax code under Taxation. Both are rules a
# human coder would learn from the codebook, and a model can be told too.

---
## 4. Codebook: define the categories

Human coders get a codebook, so let's give the model one: a short definition for each category,
and one rule for titles that fit more than one.

In [ ]:
CODEBOOK = """Categories:
- Health: health care and health insurance, Medicare and Medicaid, diseases and public health, drugs and medical devices, medical research.
- Education: schools and teachers, colleges and universities, student aid and student loans, education programs.
- Taxation: taxes of any kind, the Internal Revenue Code, tax credits and deductions, the IRS.
- Armed Forces and National Security: the military and defense, servicemembers and veterans, intelligence.
If a bill fits several categories, choose the one that matches its main purpose."""


def codebook(title):
    system = "You code US bill titles by policy area.\n\n" + CODEBOOK + "\n\nReply with the category name only."
    return [{"role": "system", "content": system},
            {"role": "user", "content": f"Bill title: {title}"}]


cb_answers, cb_labels = annotate(test, codebook, "2 Codebook")

> **✏️ Exercise 2**
>
> Improve the codebook. Based on your errors from Exercise 1, write a version with one or two
> extra **edge-case rules**, and compare it with the original on the **development** set. Keep a
> rule only if it helps there. (We leave the test set alone: Section 8 compares the strategies as
> defined above.)

In [ ]:
# ✅ Solution
MY_CODEBOOK = CODEBOOK + """
Edge cases:
- Bills about veterans, including their health care or education benefits, are Armed Forces and National Security.
- Bills that change taxes or the tax code are Taxation, even when the tax break is for health care or education."""


def my_codebook(title):
    system = "You code US bill titles by policy area.\n\n" + MY_CODEBOOK + "\n\nReply with the category name only."
    return [{"role": "system", "content": system},
            {"role": "user", "content": f"Bill title: {title}"}]


_ = annotate(dev, codebook, "Codebook (development set)", keep=False)
_ = annotate(dev, my_codebook, "Codebook + edge-case rules (development set)", keep=False)

# What to look for: rules that match how the human coders actually worked tend to help, rules
# that don't can make things worse. With only 40 development titles, a difference of one or two
# titles is noise. This is exactly how codebooks for human coders get refined, too.

---
## 5. Few-shot: show some examples

On top of the codebook, we show the model a few coded titles from the pool, as an earlier
conversation: a title from the user, the right category from the assistant. Two per category
gives eight examples.

In [ ]:
def make_few_shot(k=2, seed=SEED):
    shots = pool.groupby("policy_area").sample(k, random_state=seed).sample(frac=1, random_state=seed)

    def few_shot(title):
        messages = codebook(title)[:1]                     # the codebook system message
        for _, ex in shots.iterrows():
            messages += [{"role": "user", "content": f"Bill title: {ex['title']}"},
                         {"role": "assistant", "content": ex["policy_area"]}]
        messages.append({"role": "user", "content": f"Bill title: {title}"})
        return messages

    return few_shot, shots


few_shot, shots = make_few_shot(k=2)
shots[["title", "policy_area"]]

In [ ]:
fs_answers, fs_labels = annotate(test, few_shot, "3 Few-shot (codebook + 8 examples)")

> **✏️ Exercise 3**
>
> Do the examples matter? On the **development** set, compare few-shot with different examples
> (`make_few_shot(k=2, seed=1)`, `seed=2`, ...) and with more or fewer of them (`k=1`, `k=4`).
> How much does the score move just by swapping the examples?

In [ ]:
# ✅ Solution
for k, seed in [(2, 1), (2, 2), (2, 3), (1, SEED), (4, SEED)]:
    fs, _ = make_few_shot(k=k, seed=seed)
    annotate(dev, fs, f"Few-shot k={k}, seed={seed} (development set)", keep=False)

# What to look for: different random examples can move the score noticeably, sometimes as much as
# the difference between strategies. Few-shot works best with examples that show the BOUNDARIES
# (the tricky cases), not the obvious ones, and you should pick them on development data, never on
# the test set. More examples also make every prompt longer, so every title takes longer.

---
## 6. Chain-of-thought: reason first, then answer

We ask the model to explain its choice in a sentence or two before giving the label on a final
line. It has to write much more, so this is the slowest strategy: expect a few minutes.

In [ ]:
def chain_of_thought(title):
    system = ("You code US bill titles by policy area.\n\n" + CODEBOOK +
              "\n\nFirst explain your reasoning in one or two sentences. "
              "Then give your answer on a final line in the form 'Label: <category>'.")
    return [{"role": "system", "content": system},
            {"role": "user", "content": f"Bill title: {title}"}]


print(chat(chain_of_thought(example), max_new_tokens=120))

In [ ]:
cot_answers, cot_labels = annotate(test, chain_of_thought, "4 Chain-of-thought", max_new_tokens=120)

The explanations read well, but they are text the model generated, not a window into how it
decided. Use them to spot misunderstandings of your codebook, not as evidence that a label is
right.

---
## 7. Label scoring: read the probabilities

Instead of letting the model write, we look at the probability it gives to the **first token** of
each category name as the first token of its answer. Our four categories all start with a
different token, so that first token decides. The category with the highest probability wins.

In [ ]:
first_tokens = [tokenizer.encode(lab, add_special_tokens=False)[0] for lab in LABELS]
assert len(set(first_tokens)) == len(LABELS), "Two categories start with the same token"
print({lab: tokenizer.decode(t) for lab, t in zip(LABELS, first_tokens)})


def label_probabilities(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        next_token = model(input_ids=inputs["input_ids"],
                           attention_mask=inputs["attention_mask"]).logits[0, -1]
    return torch.softmax(next_token[first_tokens].float(), dim=-1).cpu().numpy()   # 4 numbers, sum to 1


start = time.time()
probs = np.vstack([label_probabilities(codebook(t)) for t in test["title"]])
ls_labels = pd.Series(np.array(LABELS)[probs.argmax(axis=1)], index=test.index)
results.append(score("5 Label scoring (codebook)", test, ls_labels, time.time() - start))
print(results[-1])

Two advantages: the answer is **always a valid category**, and every title gets a
**confidence**: the probability of the winning category. Most annotation tools built on hosted
APIs can't give you this, because they only return text.

> **✏️ Exercise 4**
>
> Is the model right more often when it is confident? Split the test titles into the half with
> the highest confidence (`probs.max(axis=1)`) and the half with the lowest, and compare the
> accuracy of `ls_labels` in each half. How could you use this in a real project?

In [ ]:
# ✅ Solution
confidence = probs.max(axis=1)
high = confidence >= np.median(confidence)

for name, mask in [("most confident half", high), ("least confident half", ~high)]:
    acc = accuracy_score(test["policy_area"][mask], ls_labels[mask])
    print(f"{name:<22} accuracy {acc:.2f}   ({mask.sum()} titles)")

# What to look for: accuracy is usually clearly higher among the confident half. In a real project
# you could let the model code the confident cases and send the uncertain ones to a human coder:
# you save most of the work and spend the human effort where the model is weakest. (Check this on
# your own validation data: confidence is not guaranteed to be well calibrated.)

---
## 8. Comparing the strategies

All five strategies, on the same 100 test titles, against the same human labels.

In [ ]:
summary = pd.DataFrame(results).set_index("strategy").round(3)
summary

In [ ]:
ax = summary["macro-F1"].plot(kind="barh", figsize=(8, 3), color="#34B233")
ax.set_xlabel("Test macro-F1 (agreement with the human coders)")
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
for i, v in enumerate(summary["macro-F1"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
plt.tight_layout()
plt.show()

And per category, for the strategy with the highest macro-F1:

In [ ]:
best = summary["macro-F1"].idxmax()
best_labels = {"1 Zero-shot": zs_labels, "2 Codebook": cb_labels,
               "3 Few-shot (codebook + 8 examples)": fs_labels,
               "4 Chain-of-thought": cot_labels, "5 Label scoring (codebook)": ls_labels}[best]
print("Best:", best, "\n")
print(classification_report(test["policy_area"], best_labels, labels=LABELS, digits=3, zero_division=0))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(test["policy_area"], best_labels, labels=LABELS,
                                        display_labels=[l.replace(" and ", "\n& ") for l in LABELS],
                                        cmap="Greens", colorbar=False, ax=ax)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

Some things to look for, and to discuss:

- **Which strategy won, and by how much?** With 100 test titles, one title is one percentage point
  of accuracy. Small differences may be noise.
- **Invalid answers.** A strategy that often answers outside the categories loses points for
  that alone. Label scoring can't.
- **Speed.** Compare the seconds per title. Chain-of-thought costs many times more for every
  title, which matters with 100,000 texts.
- **Compare with Day 3.** How does the best strategy do against the TF-IDF baseline and the
  fine-tuned DistilBERT model on the same task? The LLM needed no training data, but you still
  needed human labels to know how well it did.

---
## 9. Does the model agree with itself?

With human coders we check whether two coders agree. With an LLM, ask whether it agrees with
**itself**. At temperature 0 it gives the same answer every time on the same machine. With
sampling switched on, it doesn't:

In [ ]:
_, run_1 = annotate(dev, codebook, "Codebook, temperature 1, run 1", temperature=1.0, keep=False)
_, run_2 = annotate(dev, codebook, "Codebook, temperature 1, run 2", temperature=1.0, keep=False)

print(f"\nThe two runs agree on {(run_1 == run_2).mean():.0%} of the titles "
      f"(Cohen's kappa {cohen_kappa_score(run_1, run_2):.2f})")

> **✏️ Exercise 5**
>
> Repeat the comparison at temperature 0 (`temperature=0.0`) and at temperature 0.5. How does the
> agreement between two runs change? Which setting would you use for a research project, and
> what would you report about it?

In [ ]:
# ✅ Solution
for t in [0.0, 0.5]:
    _, a = annotate(dev, codebook, f"temperature {t}, run 1", temperature=t, keep=False)
    _, b = annotate(dev, codebook, f"temperature {t}, run 2", temperature=t, keep=False)
    print(f"→ temperature {t}: runs agree on {(a == b).mean():.0%} of titles\n")

# What to look for: at temperature 0 the two runs agree completely; the higher the temperature,
# the more the labels wander. For measurement, use temperature 0 (or label scoring), and still
# report it: hosted models in particular don't always give identical answers even at 0, and the
# model behind a name can change over time.

---
## 10. Other routes *(optional)*

The same codebook, through two other routes from the slides. Section 10.1 runs the same model in
Ollama; Section 10.2 a model 50 times bigger through OpenRouter. We free the GPU first.

In [ ]:
del model
torch.cuda.empty_cache()

### 10.1 Ollama

On your own laptop you would install Ollama from [ollama.com](https://ollama.com) and run
`ollama pull qwen2.5:1.5b` in a terminal. Colab is Linux, so we install it here. Ollama's version
of the model is **quantized to 4 bits**.

In [ ]:
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):                                       # wait until Ollama answers
    try:
        print(requests.get("http://localhost:11434", timeout=1).text)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

In [ ]:
!ollama pull qwen2.5:1.5b

In [ ]:
import ollama


def ask_ollama(messages):
    reply = ollama.chat(model="qwen2.5:1.5b", messages=messages, options={"temperature": 0})
    return reply.message.content.strip()


ollama_answers, ollama_labels = annotate(test, codebook, "Codebook via Ollama (4-bit)", ask=ask_ollama)

### 10.2 OpenRouter

This needs an OpenRouter account and an API key in Colab's *Secrets* (🔑) under the name
`OPENROUTER_API_KEY`. Coding 100 titles costs a fraction of a cent. Remember that the titles go to
the provider. Without a key, the cell skips itself.

In [ ]:
from openai import OpenAI

try:
    from google.colab import userdata
    OPENROUTER_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")

BIG_MODEL = "qwen/qwen-2.5-72b-instruct"     # names change: see openrouter.ai/models

if not OPENROUTER_KEY:
    print("Skipped: no OpenRouter key.")
else:
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY)

    def ask_openrouter(messages):
        reply = client.chat.completions.create(model=BIG_MODEL, messages=messages, temperature=0, max_tokens=15)
        return reply.choices[0].message.content.strip()

    annotate(test, codebook, "Codebook via OpenRouter (72B)", ask=ask_openrouter)

In [ ]:
pd.DataFrame(results).set_index("strategy").round(3)

Same codebook, three routes. Does the 4-bit version lose much against the full-precision model?
And how much does a model 50 times bigger gain? Keep in mind what you give up for it: the
titles leave your machine, and the hosted model may change or disappear.

In [ ]:
server.terminate()    # stop Ollama

---
## Wrap-up

- The same model can look good or mediocre **depending on how you ask**. Prompting is part of
  your measurement instrument, so treat it like one: design it on development data, test it once.
- Every strategy here was judged against **human labels**. Without them you wouldn't know which
  one to trust, or whether to trust any of them.
- **Report** the model and version, the route, the full prompt (codebook and examples),
  temperature, the date, and the validation scores.

**Next, Part 2:** using LLMs responsibly, and **distillation**: letting an LLM label a sample and
training a small, fast BERT classifier on those labels.

### Further reading

- Benoit, K., De Marchi, S., Laver, C., Laver, M., & Ma, J. (2026). Using large language models
  to analyze political texts through natural language understanding. *American Journal of
  Political Science*. Advance online publication. https://doi.org/10.1111/ajps.70050
- Gilardi, F., Alizadeh, M., & Kubli, M. (2023). ChatGPT outperforms crowd workers for
  text-annotation tasks. *PNAS, 120*(30).
- Rathje, S., Mirea, D.-M., Sucholutsky, I., Marjieh, R., Robertson, C. E., & Van Bavel, J. J.
  (2024). GPT is an effective tool for multilingual psychological text analysis. *PNAS, 121*(34).
- Ziems, C., Held, W., Shaikh, O., Chen, J., Zhang, Z., & Yang, D. (2024). Can large language
  models transform computational social science? *Computational Linguistics, 50*(1).
- Baumann, J., Röttger, P., Urman, A., Wendsjö, A., Plaza-del-Arco, F. M., Gruber, J. B., &
  Hovy, D. (2025). Large language model hacking: Quantifying the hidden risks of using LLMs for
  text annotation. *arXiv:2509.08825*.